In [1]:
import pandas as pd
import numpy as np
from matplotlib import pyplot as plt
import os
import seaborn as sns


%run ../funciones/functions.ipynb

In [2]:
def procesar_curvas(path_csv: str, anio: int) -> pd.DataFrame:
    """
    Procesa un archivo de curvas Tika para un año específico.
    
    Parámetros:
        path_csv (str): Ruta al archivo CSV.
        anio (int): Año del programa (ej. 2025, 2026).
    
    Retorna:
        pd.DataFrame procesado con columnas:
        ['variedad','edad','fm','prg','pico','suma_pico','puntos_pico','termina_curva']
    """
    
    # --- Leer CSV ---
    df = pd.read_csv(path_csv, thousands=',')
    
    # --- Año del programa ---
    df.loc[:,'prg'] = anio
    
    # --- Renombrar columnas ---
    rename_dict = {'Variedad': 'variedad', 'Numero de Semana': 'edad', 'Producción': 'fm'}
    df.rename(columns=rename_dict, inplace=True)
    
    # --- Selección y orden ---
    df = df[['variedad', 'edad', 'fm', 'prg']].sort_values(by=['variedad', 'edad'])
    
    
    return df


def calcular_usemana(grupo):
    grupo = grupo.reset_index(drop=True)
    
    # Revisar si hay suficientes puntos
    if len(grupo) < 4:
        grupo['pico'] = np.nan
        grupo['suma_pico'] = np.nan
        grupo['puntos_pico'] = np.nan
        grupo['termina_curva'] = np.nan
        return grupo
    
    # --- 1. Mejor ventana de 4 consecutivos ---
    rolling_sum = grupo['fm'].rolling(window=4).sum()
    idx_max_window = rolling_sum.idxmax()
    max_sum = rolling_sum.max()
    ventana = 4
    puntos = len(grupo)
    
    # Si la mejor suma de 4 puntos es < 3.2, probar con 5 puntos
    if max_sum < 3.2 and len(grupo) >= 5:
        rolling_sum = grupo['fm'].rolling(window=5).sum()
        idx_max_window = rolling_sum.idxmax()
        #max_sum = rolling_sum.max()
        ventana = 5

    edad_pico = grupo.loc[idx_max_window, 'edad']
    grupo['pico'] = edad_pico
    grupo['suma_pico'] = max_sum
    grupo['puntos_pico'] = ventana

    # --- 2. Última edad con fm > 0 ---
    mask = grupo['fm'] > 0
    if mask.any():
        ultima_edad = grupo.loc[mask, 'edad'].iloc[-1]
    else:
        ultima_edad = np.nan  # devuelve NaN si no hay fm > 0
    grupo['termina_curva'] = ultima_edad
    grupo['puntos'] = puntos
    
    return grupo


def procesar_curvas_excel(path_excel: str, anio: int, hoja: str = "curvas") -> pd.DataFrame:
    """
    Procesa un archivo Excel de curvas (formato ancho) y lo convierte a formato largo.
    
    Parámetros:
        path_excel (str): Ruta al archivo Excel.
        anio (int): Año del programa (ej. 2024).
        hoja (str): Nombre de la hoja (default 'curvas').
    
    Retorna:
        pd.DataFrame con columnas ['variedad','edad','fm','prg']
    """
    
    # --- Leer Excel y pasar de ancho a largo ---
    df = (
        pd.read_excel(path_excel, sheet_name=hoja)
        .melt(id_vars="variedad", var_name="edad", value_name="fm")
        .fillna(0)
    )
    
    # --- Limpiar ---
    df['edad'] = pd.to_numeric(df['edad'], errors="coerce")
    df = df.query("edad >= 17 and edad <= 70")
    
    # --- Año ---
    df['prg'] = anio
    
    # --- Ordenar ---
    df = df[['variedad', 'edad', 'fm', 'prg']].sort_values(by=['variedad', 'edad'])
    
    return df

def asignar_ciclo(edad):
    if edad <= 35:
        return 1
    elif edad <= 70:
        return 2
    else:
        return 3

In [3]:
df_tika25 = procesar_curvas("//svrinverpalmas/usuarios/Hilder/Data/tika-curvas2025.csv", 2025)
df_tika26 = procesar_curvas("//svrinverpalmas/usuarios/Hilder/Data/tika-curvas2026v2.csv", 2026)

df_tika = pd.concat([df_tika25, df_tika26], ignore_index=True)

df_resultado = pd.concat([df_tika], ignore_index=True)
df_resultado = df_resultado.query("edad>=17 and edad<=70")

df_resultado['ciclo'] = df_resultado['edad'].apply(asignar_ciclo)

df_resultado = (
    df_resultado
    .groupby(["prg", "variedad", "ciclo"])
    .apply(calcular_usemana)
    .reset_index()
)

In [4]:
df_resultado.sample()

,prg,variedad,ciclo,level_3,edad,fm,pico,suma_pico,puntos_pico,termina_curva,puntos
9423,2026,HONEY,2,8,44,0.0,57,1.55,5,68,35


In [5]:
# 1. Sumas por ciclo
resumen = (
    df_resultado
    .groupby(["prg", "variedad", "ciclo"])["fm"]
    .sum()
    .unstack(fill_value=0)  # columnas por ciclo
    .rename(columns={1: "suma_ciclo1", 2: "suma_ciclo2"})
    .reset_index()
)

# 2. Agregar la columna total
resumen["suma_total"] = resumen[["suma_ciclo1", "suma_ciclo2"]].sum(axis=1)
#resumen = resumen.drop(columns='suma_pico')

# 3. Extraer pico y suma_pico solo del ciclo 1
df_ppico = (
    df_resultado.query("ciclo == 1")
    .groupby(["prg", "variedad"])
    .agg({
        "pico": "first",
        "suma_pico": "first"
    })
    .reset_index()
)

df_resultado = df_resultado.drop(columns='suma_pico')

# 4. Unir todo al df original
df_export = (
    df_resultado
    .drop(columns="pico")
    .merge(resumen, on=["prg", "variedad"], how="left")
    .merge(df_ppico, on=["prg", "variedad"], how="left")
)

In [6]:
conexion = DataBase()
conexion.create_engine
conexion.convert_to_sql(df_resultado, 'resumen_curvas')
conexion.close_conexion

DataFrame convertido a la tabla 'resumen_curvas' en la base de datos


<bound method DataBase.close_conexion of <__main__.DataBase object at 0x0000027451A7A3F0>>

In [7]:
df_export.to_csv("//172.10.18.2/Usuarios/Outputs/curvas_resumen.csv")

In [18]:
#variedades y fechas
sql_curvas = """
SELECT 
    c.finca,
    c.cflor,
    c.bloque,
    REPLACE(c.nvariedad, ' ', '_') AS nvariedad,
    c.cosecha,
    c.edad,
    c.f_m,
    if(d.finca is null, 0,1) as descarta_curva
FROM
    ld_curvas_clavel as c
left join ld_descarta_curva as d
on d.finca=c.finca and d.variedad=c.variedad and d.cosecha=c.cosecha and d.bloque=c.bloque
"""

sql_metas = """
SELECT 
    finca, bloque, REPLACE(nvariedad,' ','_'), cosecha, cantidad
FROM
    ld_metas
"""

sql_cosechas = """
SELECT 
    cod_temporada, ncosecha
FROM
    seasons
"""
# Create a database instance and connect to the database
db = DataBase()
db.connect()

df_ = pd.DataFrame(db.execute_query(sql_curvas), 
                        columns=['finca', 'flor', 'bloque', 'nvariedad', 'cosecha', 'edad', 'fm', 'descarta_curva'])

# Close the database connection
df_metas = pd.DataFrame(db.execute_query(sql_metas),columns=['finca', 'bloque', 'variedad', 'cosecha','meta'])
df_metas = df_metas.drop_duplicates(subset=['finca','bloque','variedad','cosecha'],keep='last')

df_dates = pd.DataFrame(db.execute_query("SELECT aass, fecha, año, semana FROM dates WHERE dia_semana=1"), 
                        columns=['aass', 'fecha', 'año', 'semana'])
df_dates['aass'] = df_dates['aass'].astype('int')

df_cosechas = pd.DataFrame(db.execute_query(sql_cosechas),columns=['cosecha','ncosecha'])

db.close_conexion()

Conexión exitosa a la base de datos
Conexión cerrada


In [9]:
df_['ciclo'] = df_['edad'].apply(asignar_ciclo)
df_ = df_[df_.isna() != True]
df_ = df_.query("edad>=17 and edad<=70")

df_ = df_[
    (df_['flor'].isin(['CLA', 'CM0'])) & 
    (df_['cosecha'] != 'VITRI')
]

In [10]:
df_resultado2 = df_.groupby(
    ["finca","flor","bloque","nvariedad", "ciclo", "cosecha"], 
    group_keys=True).apply(calcular_usemana)

df_resultado2 = df_resultado2.reset_index()  # Limpia el índice
df_resultado2 = df_resultado2.rename(columns={'nvariedad':'variedad'})
df_resultado2_clean = df_resultado2.dropna()

In [11]:
# 1. Sumas por ciclo
resumen2 = (
    df_resultado2_clean
    .groupby(["finca","bloque", "variedad", "cosecha", "ciclo"])["fm"]
    .sum()
    .unstack(fill_value=0)  # columnas por ciclo
    .rename(columns={1: "suma_ciclo1", 2: "suma_ciclo2"})
    .reset_index()
)

# 2. Agregar la columna total
resumen2["suma_total"] = resumen2[["suma_ciclo1", "suma_ciclo2"]].sum(axis=1)
#resumen = resumen.drop(columns='suma_pico')

# 3. Extraer pico y suma_pico solo del ciclo 1
df_ppico2 = (
    df_resultado2_clean.query("ciclo == 1")
    .groupby(["finca","bloque", "variedad", "cosecha"])
    .agg({
        "pico": "first",
        "suma_pico": "first",
        "puntos": "first"
    })
    .reset_index()
)

df_resultado2_clean = df_resultado2_clean.drop(columns='suma_pico')

# 4. Unir todo al df original
df_export2 = (
    df_resultado2_clean
    .drop(columns="pico")
    .merge(resumen2, on=["finca","bloque", "variedad", "cosecha"], how="left")
    .merge(df_ppico2, on=["finca","bloque", "variedad", "cosecha"], how="left")
)

df_export2 = df_export2.drop(columns='puntos_x')
df_export2 = df_export2.rename(columns={'puntos_y':'puntos_ppico'})

In [12]:
df_meta_real = pd.merge(
    resumen2,df_metas,on=['finca','bloque','variedad','cosecha'], how='inner'
)

In [13]:
df_meta_real.loc[:,'dif'] = np.round(df_meta_real['suma_ciclo1'] - df_meta_real['meta'],2)

In [20]:
df_export3 = pd.merge(
    df_export2,df_cosechas, on='cosecha',how='left'
)

In [21]:
df_export3.to_csv("//172.10.18.2/Usuarios/Outputs/curvas_resumen_todas.csv")